# LLM Phase — Notebook 1: RAG Pipeline



## Section 1: Setup & Imports

In [ ]:
# Install required packages (run once)
%pip install anthropic sentence-transformers faiss-cpu tiktoken pypdf2 pdfplumber pytesseract pdf2image pymupdf pillow tqdm numpy pandas python-dotenv
# Note: For OCR support, you also need Tesseract installed on your system:
#   macOS:   brew install tesseract
#   Ubuntu:  sudo apt-get install tesseract-ocr
#   Windows: Download installer from https://github.com/UB-Mannheim/tesseract/wiki


In [ ]:
import os
import json
import time
import pickle
import hashlib
import re
from pathlib import Path
from typing import List, Dict, Tuple, Optional

import numpy as np
import pandas as pd
import PyPDF2
import pdfplumber
import fitz  # PyMuPDF
import tiktoken
import faiss
from tqdm import tqdm
from dotenv import load_dotenv
from anthropic import Anthropic
from sentence_transformers import SentenceTransformer

try:
    import pytesseract
    from pdf2image import convert_from_path
    from PIL import Image
    import io
    OCR_AVAILABLE = True
except ImportError:
    OCR_AVAILABLE = False
    print("Warning: OCR libraries not available. Install pytesseract, pdf2image, and pillow for OCR support.")



## Section 2: Configuration

In [ ]:
ENV_FILE = Path('/Users/roya/Desktop/Thesis/Thesis_Project/LLMs/ATT18160.env')
load_dotenv(ENV_FILE)
client = Anthropic(api_key=os.environ.get('OPENAI_API_KEY'))  

CLAUDE_MODEL         = 'claude-3-5-sonnet-20241022'  
EMBED_MODEL          = 'all-MiniLM-L6-v2'            
EMBEDDING_DIMENSIONS = 384                           
CHUNK_SIZE_TOKENS    = 512    
CHUNK_OVERLAP_TOKENS = 64     

st_model = SentenceTransformer(EMBED_MODEL)

PROJECT_ROOT   = Path('/Users/roya/Desktop/Thesis')
RAG_DOCS_DIR   = PROJECT_ROOT / 'RAG documents'
OUTPUT_DIR     = Path('/Users/roya/Desktop/Thesis/Thesis_Project/LLMs/outputs')
VECTOR_DB_DIR  = OUTPUT_DIR / 'vector_db'
VECTOR_DB_DIR.mkdir(parents=True, exist_ok=True)

FAISS_INDEX_PATH   = VECTOR_DB_DIR / 'medicare_policy.faiss'
METADATA_PATH      = VECTOR_DB_DIR / 'chunk_metadata.pkl'
CHUNK_TEXTS_PATH   = VECTOR_DB_DIR / 'chunk_texts.pkl'

print(f'RAG documents directory: {RAG_DOCS_DIR}')
print(f'Vector DB output:        {VECTOR_DB_DIR}')
print(f'Embedding model:         {EMBED_MODEL} ({EMBEDDING_DIMENSIONS}-dim, local/free)')



## Section 3: PDF Document Loading


In [ ]:
def extract_text_method1_pypdf2(pdf_path: Path) -> Tuple[str, str]:
    
    text_parts = []
    try:
        with open(pdf_path, 'rb') as f:
            reader = PyPDF2.PdfReader(f)
            for page_num, page in enumerate(reader.pages):
                page_text = page.extract_text()
                if page_text and page_text.strip():
        
                    page_text = re.sub(r'\s+', ' ', page_text)  
                    page_text = page_text.strip()
                    text_parts.append(f'[PAGE {page_num + 1}] {page_text}')
    except Exception as e:
        return '', f'method1_error: {e}'
    return ' '.join(text_parts), 'method1_pypdf2'


def extract_text_method2_pdfplumber(pdf_path: Path) -> Tuple[str, str]:
   
    text_parts = []
    try:
        with pdfplumber.open(pdf_path) as pdf:
            for page_num, page in enumerate(pdf.pages):
            
                page_text = page.extract_text()
                if page_text and page_text.strip():
                    
                    page_text = re.sub(r'\s+', ' ', page_text)
                    page_text = page_text.strip()
                    text_parts.append(f'[PAGE {page_num + 1}] {page_text}')
                    
                
                tables = page.extract_tables()
                for table in tables:
                    if table:
                       
                        table_text = '\n'.join([' | '.join([str(cell) if cell else '' for cell in row]) for row in table])
                        if table_text.strip():
                            text_parts.append(f'[TABLE] {table_text}')
    except Exception as e:
        return '', f'method2_error: {e}'
    return ' '.join(text_parts), 'method2_pdfplumber'


def extract_text_method3_ocr(pdf_path: Path) -> Tuple[str, str]:
    
    if not OCR_AVAILABLE:
        return '', 'method3_ocr_unavailable'
    
    text_parts = []
    try:
        
        pdf_document = fitz.open(pdf_path)
        
        for page_num in range(len(pdf_document)):
            page = pdf_document[page_num]
            
            
            page_text = page.get_text()
            
            
            if len(page_text.strip()) < 50:
                
                pix = page.get_pixmap(matrix=fitz.Matrix(300/72, 300/72))
                img_data = pix.tobytes("png")
                
                
                img = Image.open(io.BytesIO(img_data))
                page_text = pytesseract.image_to_string(img)
            
            if page_text and page_text.strip():
                page_text = re.sub(r'\s+', ' ', page_text)
                page_text = page_text.strip()
                text_parts.append(f'[PAGE {page_num + 1}] {page_text}')
        
        pdf_document.close()
    except Exception as e:
        return '', f'method3_error: {e}'
    return ' '.join(text_parts), 'method3_ocr'


def extract_text_from_pdf(pdf_path: Path) -> Tuple[str, str]:
   
    MIN_CHARS_THRESHOLD = 100  
    OCR_THRESHOLD = 50  
    text1, method1 = extract_text_method1_pypdf2(pdf_path)
    text2, method2 = extract_text_method2_pdfplumber(pdf_path)
    
    results = [
        (text1, method1, len(text1.strip())),
        (text2, method2, len(text2.strip()))
    ]
    
    if results[0][2] < OCR_THRESHOLD and results[1][2] < OCR_THRESHOLD:
        text3, method3 = extract_text_method3_ocr(pdf_path)
        results.append((text3, method3, len(text3.strip())))
    
    best_text, best_method, best_length = max(results, key=lambda x: x[2])
    
    if best_length < MIN_CHARS_THRESHOLD:
        return best_text, f'{best_method}_low_quality'
    
    return best_text, best_method


def categorize_document(pdf_path: Path) -> str:
    path_str = str(pdf_path)
    if 'Claims Processing' in path_str:
        return 'Medicare Claims Processing Manual'
    elif 'Program Integrity' in path_str:
        return 'Medicare Program Integrity Manual'
    elif 'OIG' in path_str and 'Inspector' in path_str:
        return 'OIG Fraud Alerts'
    else:
        return 'Medicare Policy'


def load_all_pdf_documents(rag_docs_dir: Path) -> List[Dict]:
    
    documents = []
    pdf_files = sorted(rag_docs_dir.rglob('*.pdf'))
    extraction_stats = {'method1_pypdf2': 0, 'method2_pdfplumber': 0, 'method3_ocr': 0, 'failed': 0}

    print(f'Found {len(pdf_files)} PDF files. Extracting text using multi-method approach...')

    for pdf_path in tqdm(pdf_files, desc='Loading PDFs'):
        text, method = extract_text_from_pdf(pdf_path)
        
        if len(text.strip()) < 100:  
            extraction_stats['failed'] += 1
            continue
        
        if 'method1' in method:
            extraction_stats['method1_pypdf2'] += 1
        elif 'method2' in method:
            extraction_stats['method2_pdfplumber'] += 1
        elif 'method3' in method:
            extraction_stats['method3_ocr'] += 1
        
        documents.append({
            'text': text,
            'source_file': pdf_path.name,
            'source_path': str(pdf_path.relative_to(rag_docs_dir)),
            'source_category': categorize_document(pdf_path),
            'file_size_kb': round(pdf_path.stat().st_size / 1024, 1),
            'extraction_method': method
        })

    total_chars = sum(len(d['text']) for d in documents)
    print(f'\nLoaded {len(documents)} documents, {total_chars:,} total characters.')
    print('\nExtraction method statistics:')
    print(f'  Method 1 (PyPDF2):       {extraction_stats["method1_pypdf2"]:>3} PDFs')
    print(f'  Method 2 (pdfplumber):   {extraction_stats["method2_pdfplumber"]:>3} PDFs')
    print(f'  Method 3 (OCR):          {extraction_stats["method3_ocr"]:>3} PDFs')
    if extraction_stats['failed'] > 0:
        print(f'  Failed/Empty:            {extraction_stats["failed"]:>3} PDFs')
    return documents


In [ ]:
documents = load_all_pdf_documents(RAG_DOCS_DIR)

from collections import Counter
category_counts = Counter(d['source_category'] for d in documents)
print('\nDocuments by category:')
for cat, count in category_counts.most_common():
    print(f'  {cat}: {count} files')


## Section 4: Text Chunking


In [ ]:
_PAGE_MARKER_RE = re.compile(r'\[PAGE\s+(\d+)\]')


_CHAPTER_RE = re.compile(
    r'(?:Chapter\s+\d+[\w\s,\-–—]{0,80}|'  
    r'\b\d{1,3}\s*[-–—]\s*[A-Z][^\n]{3,60})', 
    re.IGNORECASE
)


_SECTION_RE = re.compile(
    r'(?:Section\s+[\d.]+[^\n.]{0,60}|'       
    r'§\s*[\d.]+[^\n.]{0,60}|'                
    r'\d+(?:\.\d+)+\s*[-–—]\s*\w[^\n]{3,60})', 
    re.IGNORECASE
)


def extract_page_range(chunk_text: str) -> Dict[str, Optional[int]]:
    pages = [int(m) for m in _PAGE_MARKER_RE.findall(chunk_text)]
    return {
        'start_page': pages[0]  if pages else None,
        'end_page':   pages[-1] if pages else None,
    }


def extract_section_context(chunk_text: str) -> Dict[str, Optional[str]]:

    ch_match  = _CHAPTER_RE.search(chunk_text)
    sec_match = _SECTION_RE.search(chunk_text)
    return {
        'chapter':    ch_match.group(0).strip()  if ch_match  else None,
        'subsection': sec_match.group(0).strip() if sec_match else None,
    }




def count_tokens(text: str, encoding_name: str = 'cl100k_base') -> int:
    enc = tiktoken.get_encoding(encoding_name)
    return len(enc.encode(text))


def split_text_into_chunks(
    text: str,
    chunk_size: int = CHUNK_SIZE_TOKENS,
    overlap: int = CHUNK_OVERLAP_TOKENS
) -> List[str]:
   
    enc = tiktoken.get_encoding('cl100k_base')
    tokens = enc.encode(text)

    chunks = []
    start = 0
    while start < len(tokens):
        end = min(start + chunk_size, len(tokens))
        chunk_tokens = tokens[start:end]
        chunk_text = enc.decode(chunk_tokens)
        chunks.append(chunk_text.strip())
        if end == len(tokens):
            break
        start += (chunk_size - overlap)

    return [c for c in chunks if len(c.strip()) > 50]


def chunk_all_documents(documents: List[Dict]) -> Tuple[List[str], List[Dict]]:
    
    chunk_texts = []
    chunk_metadata = []

    for doc_idx, doc in enumerate(tqdm(documents, desc='Chunking documents')):
        chunks = split_text_into_chunks(doc['text'])
        for chunk_idx, chunk_text in enumerate(chunks):
            
            page_range  = extract_page_range(chunk_text)
            section_ctx = extract_section_context(chunk_text)

            
            clean_text = _PAGE_MARKER_RE.sub('', chunk_text)
            clean_text = re.sub(r'\s{2,}', ' ', clean_text).strip()

            chunk_texts.append(clean_text)
            chunk_metadata.append({
                'doc_id':          doc_idx,
                'chunk_idx':       chunk_idx,
                'total_chunks':    len(chunks),
                'source_file':     doc['source_file'],
                'source_path':     doc['source_path'],
                'source_category': doc['source_category'],
                'start_page':      page_range['start_page'],
                'end_page':        page_range['end_page'],
                'chapter':         section_ctx['chapter'],
                'subsection':      section_ctx['subsection'],
                'chunk_id':        f"{doc['source_file']}__chunk_{chunk_idx:04d}"
            })

    print(f'Created {len(chunk_texts):,} chunks from {len(documents)} documents.')
    return chunk_texts, chunk_metadata


In [ ]:
chunk_texts, chunk_metadata = chunk_all_documents(documents)

cat_chunk_counts = Counter(m['source_category'] for m in chunk_metadata)
print('\nChunks by source category:')
for cat, count in cat_chunk_counts.most_common():
    print(f'  {cat}: {count:,} chunks')

print(f'\nSample chunk (chunk #100):')
print(f'  Source: {chunk_metadata[100]["source_file"]}')
print(f'  Category: {chunk_metadata[100]["source_category"]}')
print(f'  Text (first 300 chars): {chunk_texts[100][:300]}...')


## Section 5: Embedding Generation


In [ ]:
def generate_embeddings_batch(
    texts: List[str],
    batch_size: int = 64
) -> np.ndarray:
    
    texts = [t if t.strip() else 'empty' for t in texts]
    embeddings = st_model.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True
    )
    return embeddings.astype(np.float32)


def embed_query(query: str) -> np.ndarray:
    vec = st_model.encode([query], convert_to_numpy=True)[0]
    return vec.astype(np.float32)


In [ ]:

print(f'Generating embeddings for {len(chunk_texts):,} chunks...')
print(f'Using model: {EMBED_MODEL} ({EMBEDDING_DIMENSIONS}-dim, local)')
embeddings = generate_embeddings_batch(chunk_texts)

print(f'\nEmbeddings shape: {embeddings.shape}')
print(f'Memory usage: {embeddings.nbytes / 1024**2:.1f} MB')



## Section 6: FAISS Vector Database


In [ ]:
def build_faiss_index(embeddings: np.ndarray) -> faiss.Index:
   
    norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
    normalized = embeddings / (norms + 1e-10)
    
    dim = normalized.shape[1]
    index = faiss.IndexFlatIP(dim)
    index.add(normalized)
    
    print(f'FAISS index built: {index.ntotal:,} vectors, {dim} dimensions')
    return index, normalized


def save_vector_database(
    index: faiss.Index,
    chunk_texts: List[str],
    chunk_metadata: List[Dict],
    normalized_embeddings: np.ndarray
):
    
    faiss.write_index(index, str(FAISS_INDEX_PATH))
    with open(CHUNK_TEXTS_PATH, 'wb') as f:
        pickle.dump(chunk_texts, f)
    with open(METADATA_PATH, 'wb') as f:
        pickle.dump(chunk_metadata, f)
    np.save(VECTOR_DB_DIR / 'normalized_embeddings.npy', normalized_embeddings)
    print(f'Saved vector database to: {VECTOR_DB_DIR}')


def load_vector_database() -> Tuple[faiss.Index, List[str], List[Dict]]:
    index = faiss.read_index(str(FAISS_INDEX_PATH))
    with open(CHUNK_TEXTS_PATH, 'rb') as f:
        chunk_texts = pickle.load(f)
    with open(METADATA_PATH, 'rb') as f:
        chunk_metadata = pickle.load(f)
    print(f'Loaded vector database: {index.ntotal:,} vectors')
    return index, chunk_texts, chunk_metadata

In [ ]:
index, normalized_embeddings = build_faiss_index(embeddings)
save_vector_database(index, chunk_texts, chunk_metadata, normalized_embeddings)


## Section 7: Retrieval Function


In [ ]:
def retrieve(
    query: str,
    index: faiss.Index,
    chunk_texts: List[str],
    chunk_metadata: List[Dict],
    top_k: int = 5
) -> List[Dict]:
    query_vec = embed_query(query)
    query_vec = query_vec / (np.linalg.norm(query_vec) + 1e-10)
    query_vec = query_vec.reshape(1, -1)

    scores, indices = index.search(query_vec, top_k)

    results = []
    for rank, (score, idx) in enumerate(zip(scores[0], indices[0]), start=1):
        if idx == -1:  
            continue
        meta = chunk_metadata[idx]
        results.append({
            'rank':            rank,
            'score':           float(score),
            'text':            chunk_texts[idx],
            'source_file':     meta['source_file'],
            'source_category': meta['source_category'],
            'start_page':      meta.get('start_page'),
            'end_page':        meta.get('end_page'),
            'chapter':         meta.get('chapter'),
            'subsection':      meta.get('subsection'),
            'chunk_id':        meta['chunk_id']
        })
    return results



## Section 8: Retrieval Quality Tests


In [ ]:
test_queries = [
    'What constitutes Medicare fraud in physician billing?',
    'Indicators of fraudulent prescription drug claims in Part D',
    'OIG guidelines for identifying excluded providers',
    'Upcoding and unbundling in Medicare claims',
    'High volume billing patterns associated with fraud',
]

for query in test_queries:
    print(f'\n{"="*70}')
    print(f'Query: {query}')
    print(f'{"="*70}')
    results = retrieve(query, index, chunk_texts, chunk_metadata, top_k=3)
    for r in results:
        if r['start_page'] and r['end_page'] and r['start_page'] != r['end_page']:
            pages_str = f"pp. {r['start_page']}–{r['end_page']}"
        elif r['start_page']:
            pages_str = f"p. {r['start_page']}"
        else:
            pages_str = 'page unknown'

        chapter_str    = (r['chapter']    or 'n/a')[:60]
        subsection_str = (r['subsection'] or 'n/a')[:60]

        print(f"  [{r['rank']}] Score={r['score']:.4f} | {r['source_category']} | {r['source_file']}")
        print(f"      Pages:      {pages_str}")
        print(f"      Chapter:    {chapter_str}")
        print(f"      Subsection: {subsection_str}")
        print(f"      Excerpt:    {r['text'][:200]}...")



## Section 9: Corrective RAG (CRAG) — Relevance Filtering



In [ ]:
def check_chunk_relevance(
    query: str,
    chunk_text: str,
    threshold: float = 0.6
) -> Tuple[bool, float]:
    
    prompt = (
        f'You are evaluating whether a retrieved document chunk is relevant to a query.\n\n'
        f'Query: {query}\n\n'
        f'Document Chunk (first 300 chars):\n{chunk_text[:300]}\n\n'
        f'Rate the relevance on a scale from 0.0 to 1.0.\n'
        f'Respond with ONLY a single number, e.g. 0.85'
    )
    try:
        response = client.messages.create(
            model=CLAUDE_MODEL,
            system='You are evaluating document relevance. Respond with ONLY a single decimal number between 0.0 and 1.0.',
            messages=[{'role': 'user', 'content': prompt}],
            max_tokens=10,
            temperature=0
        )
        score_str = response.content[0].text.strip()
        score = float(re.search(r'[0-9.]+', score_str).group())
        score = max(0.0, min(1.0, score))
        return score >= threshold, score
    except Exception:
        return True, 0.5 

def corrective_retrieve(
    query: str,
    index: faiss.Index,
    chunk_texts: List[str],
    chunk_metadata: List[Dict],
    top_k: int = 5,
    relevance_threshold: float = 0.5,
    use_crag: bool = True
) -> Tuple[List[Dict], List[Dict]]:
    
    candidates = retrieve(query, index, chunk_texts, chunk_metadata, top_k=top_k)

    if not use_crag:
        return candidates, []

    accepted, rejected = [], []
    for chunk in candidates:
        is_relevant, score = check_chunk_relevance(query, chunk['text'], relevance_threshold)
        chunk['relevance_score'] = score
        if is_relevant:
            accepted.append(chunk)
        else:
            rejected.append(chunk)

    return accepted, rejected


In [ ]:
test_query = 'What are the OIG guidelines for identifying providers with abnormally high billing?'
accepted, rejected = corrective_retrieve(
    test_query, index, chunk_texts, chunk_metadata,
    top_k=5, use_crag=True
)

print(f'Query: {test_query}')
print(f'Accepted chunks: {len(accepted)}')
print(f'Rejected chunks: {len(rejected)}')
for r in accepted:
    if r['start_page'] and r['end_page'] and r['start_page'] != r['end_page']:
        pages_str = f"pp. {r['start_page']}–{r['end_page']}"
    elif r['start_page']:
        pages_str = f"p. {r['start_page']}"
    else:
        pages_str = 'page unknown'
    chapter_str    = (r['chapter']    or 'n/a')[:50]
    subsection_str = (r['subsection'] or 'n/a')[:50]
    print(f"  ✓ Score={r['score']:.4f} | CRAG={r['relevance_score']:.2f} | {r['source_category']} | {r['source_file']}")
    print(f"    Pages: {pages_str} | Chapter: {chapter_str} | Subsection: {subsection_str}")



## Section 10: Summary Statistics

In [ ]:
print('=' * 55)
print('RAG KNOWLEDGE BASE — BUILD COMPLETE')
print('=' * 55)
print(f'Documents loaded     : {len(documents):>6}')
print(f'Chunks created       : {len(chunk_texts):>6,}')
print(f'Embedding dimensions : {EMBEDDING_DIMENSIONS:>6}')
print(f'FAISS index vectors  : {index.ntotal:>6,}')
print()
print('Output files:')
for f in VECTOR_DB_DIR.iterdir():
    size_mb = f.stat().st_size / 1024**2
    print(f'  {f.name:<40} {size_mb:.1f} MB')
print()
print('These files are used by Notebooks 2, 3, and 4.')